# 08 — Plan and Solve Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Plan and Solve vs Zero-Shot CoT

In [3]:
problem = "Lisa has 3 bags of marbles. Each bag has 8 red and 5 blue marbles. She gives away 2 full bags to friends. How many marbles does she have left?"

zs_cot = chat([{"role":"user","content": problem + "\nLet\'s think step by step."}],
              temperature=0, max_tokens=200)

ps_prompt = (
    problem + "\n\n"
    "First, let\'s create a plan:\n"
    "1. Find total marbles per bag.\n"
    "2. Find total marbles in all 3 bags.\n"
    "3. Calculate marbles given away.\n"
    "4. Subtract to find remaining.\n\n"
    "Now solve each step:"
)
ps = chat([{"role":"user","content":ps_prompt}], temperature=0, max_tokens=200)

print("Zero-Shot CoT:")
print(zs_cot.strip())
print("\n" + "="*50)
print("Plan and Solve:")
print(ps.strip())


Zero-Shot CoT:
To find out how many marbles Lisa has left, let's break it down step by step.

1. Lisa starts with 3 bags of marbles. Each bag has 8 red and 5 blue marbles. So, the total number of marbles in each bag is 8 (red) + 5 (blue) = 13 marbles.

2. Since Lisa has 3 bags, the total number of marbles she has initially is 3 * 13 = 39 marbles.

3. Lisa gives away 2 full bags to her friends. Each bag has 13 marbles, so she gives away 2 * 13 = 26 marbles.

4. To find out how many marbles Lisa has left, we subtract the number of marbles she gave away from the total number of marbles she initially had: 39 - 26 = 13 marbles.

So, Lisa has 13 marbles left.

Plan and Solve:
Let's follow the plan to find out how many marbles Lisa has left.

**Step 1: Find total marbles per bag.**
Each bag has 8 red and 5 blue marbles. 
Total marbles per bag = 8 (red) + 5 (blue) = 13 marbles per bag.

**Step 2: Find total marbles in all 3 bags.**
Total marbles in all 3 bags = 13 (marbles per bag) * 3 (number

## Two-Stage Plan and Solve

In [4]:
task = "Write a Python function that validates an email address using regex."

plan_prompt = f"Task: {task}\n\nCreate a numbered step-by-step plan to accomplish this task. Do not write code yet."
plan = chat([{"role":"user","content":plan_prompt}], temperature=0, max_tokens=200)
print("PLAN:")
print(plan.strip())


PLAN:
Here's a step-by-step plan to create a Python function that validates an email address using regex:

1. **Define the requirements**: Identify the characteristics of a valid email address, including:
	* The presence of a username (local part)
	* The presence of a domain name
	* The use of the "@" symbol to separate the local part from the domain
	* The use of a top-level domain (TLD) to identify the domain
	* The use of valid characters in the local part and domain

2. **Research existing email address regex patterns**: Look for established regex patterns that match the requirements of a valid email address. Some common patterns include:
	* The "simple" pattern: `^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$`
	* The "RFC 5322" pattern: `^(?:[a-z0-9!#$%&'*


In [5]:
execute_prompt = f"Task: {task}\n\nPlan:\n{plan}\n\nNow implement the plan. Write the Python code:"
code_result = chat([{"role":"user","content":execute_prompt}], temperature=0, max_tokens=300)
print("EXECUTION:")
print(code_result.strip())


EXECUTION:
**Email Validation Function using Regex**

### Step 1: Define the Requirements

A valid email address consists of:

* A username (local part) that can contain alphanumeric characters, dots, underscores, percent signs, plus signs, and hyphens.
* A domain name that can contain alphanumeric characters, dots, and hyphens.
* The "@" symbol to separate the local part from the domain.
* A top-level domain (TLD) to identify the domain.

### Step 2: Research Existing Email Address Regex Patterns

We will use the "simple" pattern as a starting point, which is widely accepted and easy to understand.

### Step 3: Implement the Email Validation Function

```python
import re

def validate_email(email):
    """
    Validate an email address using a regex pattern.

    Args:
        email (str): The email address to validate.

    Returns:
        bool: True if the email is valid, False otherwise.
    """
    # Define the regex pattern for a valid email address
    pattern = r"^[a-zA-Z0-9._

## Plan and Solve for Complex Analysis

In [6]:
analysis_task = "Analyze the pros and cons of remote work for a software company."

ps_analysis = (
    f"{analysis_task}\n\n"
    "Plan:\n"
    "1. Identify key benefits of remote work.\n"
    "2. Identify key challenges of remote work.\n"
    "3. Consider impact on productivity, culture, and hiring.\n"
    "4. Synthesize into a balanced conclusion.\n\n"
    "Execute the plan:"
)
result = chat([{"role":"user","content":ps_analysis}], temperature=0.3, max_tokens=400)
print(result.strip())


**Plan: Analyze the Pros and Cons of Remote Work for a Software Company**

**1. Identify Key Benefits of Remote Work:**

Remote work offers several benefits to software companies, including:

- **Increased flexibility**: Employees can work from anywhere, at any time, as long as they have a stable internet connection. This flexibility allows for better work-life balance and increased productivity.
- **Cost savings**: Companies can save on office space, utilities, and other overhead costs associated with maintaining a physical workspace.
- **Broader talent pool**: By allowing remote work, companies can attract and hire the best talent from all over the world, rather than being limited to a specific geographic area.
- **Improved employee satisfaction**: Employees can work in a comfortable and familiar environment, which can lead to increased job satisfaction and reduced turnover rates.
- **Reduced distractions**: Some employees may find that they are more productive when working from home

## Decompose-Then-Solve

In [7]:
complex_q = "What programming language should a startup use for a web app, and how long would it take a junior developer to learn it?"

decompose = (
    f"Question: {complex_q}\n\n"
    "Sub-questions:\n"
    "1. What are the best languages for web app development for startups?\n"
    "2. What factors determine the right choice?\n"
    "3. What is the learning curve for each top candidate?\n\n"
    "Answer each sub-question, then synthesize:"
)
result = chat([{"role":"user","content":decompose}], temperature=0, max_tokens=400)
print(result.strip())


**Sub-question 1: What are the best languages for web app development for startups?**

For web app development, startups often consider languages that are:

1. **JavaScript**: Used for client-side scripting, JavaScript is a popular choice for web development. It's used in frameworks like React, Angular, and Vue.js, which are ideal for building complex web applications.
2. **Python**: Known for its simplicity and versatility, Python is a popular choice for web development. It's used in frameworks like Django and Flask, which are ideal for building scalable and maintainable web applications.
3. **Ruby**: Ruby is a dynamic language that's well-suited for web development. It's used in the Ruby on Rails framework, which is ideal for building complex web applications quickly.
4. **Go**: Go (also known as Golang) is a modern language that's gaining popularity for web development. It's used in frameworks like Revel and Gin, which are ideal for building scalable and concurrent web applications.